In [2]:
%matplotlib widget

import io
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display
from contextlib import redirect_stdout
from matplotlib.patches import Circle

from pyLIMA import event, telescopes
from pyLIMA.models import FSPL_model


# ============================================================
# Helpers
# ============================================================

def _arr(x):
    return np.asarray(getattr(x, "value", x))


def add_direction_arrows(ax, x, y, color, n_arrows=4):
    """
    Agrega flechas a una trayectoria para indicar
    el sentido creciente del tiempo.
    """

    x = _arr(x)
    y = _arr(y)

    if len(x) < 3:
        return

    idxs = np.linspace(
        1,
        len(x) - 2,
        n_arrows,
        dtype=int
    )

    for idx in idxs:

        dx = x[idx + 1] - x[idx - 1]
        dy = y[idx + 1] - y[idx - 1]

        norm = np.hypot(dx, dy)

        if norm == 0:
            continue

        scale = 0.08

        ax.annotate(
            "",
            xy=(
                x[idx] + scale * dx / norm,
                y[idx] + scale * dy / norm
            ),
            xytext=(
                x[idx],
                y[idx]
            ),
            arrowprops=dict(
                arrowstyle="->",
                mutation_scale=12,
                color=color,
                lw=1.3
            )
        )


# ============================================================
# Evento ficticio
# ============================================================

def build_sim_event(
    t,
    ra=170.0,
    dec=-70.0,
    mag0=19.0,
    emag=1e-6,
    filt="G"
):
    """
    Construye un Event + Telescope.

    La curva ficticia NO se usa como datos para ajustar nada.
    Solamente necesitamos:

        1. una grilla temporal;
        2. RA/Dec para calcular paralaje;
        3. un Telescope para que pyLIMA evalúe el modelo.
    """

    ev = event.Event()

    ev.name = "FSPL parallax comparison"
    ev.ra = float(ra)
    ev.dec = float(dec)

    lc = np.c_[
        t,
        np.full_like(t, mag0),
        np.full_like(t, emag)
    ]

    tel = telescopes.Telescope(
        name="Simulation",
        camera_filter=filt,
        lightcurve=lc.astype(float),

        lightcurve_names=[
            "time",
            "mag",
            "err_mag"
        ],

        lightcurve_units=[
            "JD",
            "mag",
            "mag"
        ],

        # Earth center.
        # Esto es suficiente para estudiar principalmente
        # el paralaje anual.
        location="Earth"
    )

    # FSPL usa ld_gamma.
    # Dejamos fuente uniforme.
    tel.ld_gamma = 0.0

    ev.telescopes.append(tel)

    return ev


# ============================================================
# Construcción robusta de pyLIMA_parameters
# ============================================================

def make_pylima_parameters(model, values):
    """
    Construye el vector siguiendo el orden REAL
    de model.model_dictionnary.

    Esto evita asumir manualmente el orden de parámetros.

    Los parámetros de flujo quedan en None porque solamente
    usamos model_magnification() y sources_trajectory().
    """

    vector = []

    for parameter_name in model.model_dictionnary.keys():

        value = values.get(
            parameter_name,
            None
        )

        vector.append(value)

    return model.compute_pyLIMA_parameters(vector)


# ============================================================
# Cálculo de los dos modelos
# ============================================================

def compute_fspl_case(
    t0=2461000.0,
    u0=0.10,
    tE=100.0,
    rho=0.01,
    piEN=0.20,
    piEE=0.10,
    ra=170.0,
    dec=-70.0,
    window_k=4.0,
    n_points=4000
):
    """
    Compara:

    MODEL 1
    -------
    FSPL sin paralaje:

        t0, u0, tE, rho

    MODEL 2
    -------
    FSPL con paralaje Full:

        t0, u0, tE, rho, piEN, piEE

    No se realiza ningún ajuste.
    """

    # ---------------------------------------------------------
    # Grilla temporal
    # ---------------------------------------------------------

    t = np.linspace(
        t0 - window_k * tE,
        t0 + window_k * tE,
        int(n_points)
    )

    # ---------------------------------------------------------
    # Evento
    # ---------------------------------------------------------

    ev = build_sim_event(
        t=t,
        ra=ra,
        dec=dec
    )

    tel = ev.telescopes[0]

    # ---------------------------------------------------------
    # FSPL SIN paralaje
    # ---------------------------------------------------------

    model_no_parallax = FSPL_model.FSPLmodel(
        ev,
        parallax=["None", t0]
    )

    values_no_parallax = {
        "t0": t0,
        "u0": u0,
        "tE": tE,
        "rho": rho,
    }

    params_no_parallax = make_pylima_parameters(
        model_no_parallax,
        values_no_parallax
    )

    # ---------------------------------------------------------
    # FSPL CON paralaje
    # ---------------------------------------------------------
    #
    # La construcción del modelo hace que pyLIMA calcule
    # las efemérides/proyecciones correspondientes.
    #
    # redirect_stdout evita llenar el widget con:
    #
    # "Parallax(Full) estimated ... SUCCESS"
    # ---------------------------------------------------------

    with redirect_stdout(io.StringIO()):

        model_parallax = FSPL_model.FSPLmodel(
            ev,
            parallax=["Full", t0]
        )

    values_parallax = {
        "t0": t0,
        "u0": u0,
        "tE": tE,
        "rho": rho,
        "piEN": piEN,
        "piEE": piEE,
    }

    params_parallax = make_pylima_parameters(
        model_parallax,
        values_parallax
    )

    # ---------------------------------------------------------
    # Magnificaciones
    # ---------------------------------------------------------

    A_no = model_no_parallax.model_magnification(
        tel,
        params_no_parallax
    )

    A_par = model_parallax.model_magnification(
        tel,
        params_parallax
    )

    # ---------------------------------------------------------
    # Trayectorias
    # ---------------------------------------------------------

    traj_no = model_no_parallax.sources_trajectory(
        tel,
        params_no_parallax,
        data_type="photometry"
    )

    traj_par = model_parallax.sources_trajectory(
        tel,
        params_parallax,
        data_type="photometry"
    )

    x_no, y_no = traj_no[0], traj_no[1]
    x_par, y_par = traj_par[0], traj_par[1]

    return dict(
        t=t,

        A_no=A_no,
        A_par=A_par,

        x_no=_arr(x_no),
        y_no=_arr(y_no),

        x_par=_arr(x_par),
        y_par=_arr(y_par),

        params_no=params_no_parallax,
        params_par=params_parallax,

        model_no=model_no_parallax,
        model_par=model_parallax,

        event=ev,
        telescope=tel
    )


# ============================================================
# Plot
# ============================================================

def plot_fspl_parallax_widget(
    t0=2461000.0,
    u0=0.10,
    tE=100.0,
    rho=0.01,
    piEN=0.20,
    piEE=0.10,
    ra=170.0,
    dec=-70.0,
    window_k=4.0,
    n_points=4000,
    traj_lim=2.0,
    logA=False,
    show_source=True
):

    plt.close("all")

    # ---------------------------------------------------------
    # Calcular modelos
    # ---------------------------------------------------------

    out = compute_fspl_case(
        t0=t0,
        u0=u0,
        tE=tE,
        rho=rho,
        piEN=piEN,
        piEE=piEE,
        ra=ra,
        dec=dec,
        window_k=window_k,
        n_points=n_points
    )

    t = out["t"]

    A_no = _arr(out["A_no"])
    A_par = _arr(out["A_par"])

    x_no = out["x_no"]
    y_no = out["y_no"]

    x_par = out["x_par"]
    y_par = out["y_par"]

    # Para que el eje temporal sea legible
    dt = t - t0

    # ---------------------------------------------------------
    # Figura
    # ---------------------------------------------------------

    fig = plt.figure(
        figsize=(13, 6.5)
    )

    gs = fig.add_gridspec(
        1,
        2,
        width_ratios=[1.45, 1.0],
        wspace=0.28
    )

    axA = fig.add_subplot(
        gs[0, 0]
    )

    axT = fig.add_subplot(
        gs[0, 1]
    )

    # ========================================================
    # MAGNIFICACIÓN
    # ========================================================

    if logA:

        axA.plot(
            dt,
            np.log10(A_no),
            lw=2,
            label="FSPL — no parallax"
        )

        axA.plot(
            dt,
            np.log10(A_par),
            "--",
            lw=2,
            label="FSPL — parallax"
        )

        axA.set_ylabel(
            r"$\log_{10} A(t)$"
        )

    else:

        axA.plot(
            dt,
            A_no,
            lw=2,
            label="FSPL — no parallax"
        )

        axA.plot(
            dt,
            A_par,
            "--",
            lw=2,
            label="FSPL — parallax"
        )

        axA.set_ylabel(
            r"$A(t)$"
        )

    axA.axvline(
        0,
        color="k",
        ls=":",
        lw=1.2,
        alpha=0.6
    )

    axA.set_xlabel(
        r"$t-t_0$ [days]"
    )

    axA.set_title(
        "FSPL magnification"
    )

    axA.legend(
        frameon=False
    )

    axA.grid(
        alpha=0.25
    )

    # ---------------------------------------------------------
    # Caja de parámetros
    # ---------------------------------------------------------

    piE = np.hypot(
        piEN,
        piEE
    )

    parameter_text = (
        rf"$t_0={t0:.1f}$" "\n"
        rf"$u_0={u0:.4g}$" "\n"
        rf"$t_E={tE:.4g}\,\mathrm{{d}}$" "\n"
        rf"$\rho={rho:.3g}$" "\n"
        rf"$\pi_{{E,N}}={piEN:.3g}$" "\n"
        rf"$\pi_{{E,E}}={piEE:.3g}$" "\n"
        rf"$\pi_E={piE:.3g}$"
    )

    axA.text(
        0.03,
        0.97,
        parameter_text,

        transform=axA.transAxes,

        ha="left",
        va="top",

        fontsize=10,

        bbox=dict(
            boxstyle="round,pad=0.35",
            fc="white",
            ec="0.5",
            alpha=0.90
        )
    )

    # ========================================================
    # TRAYECTORIAS
    # ========================================================

    # Einstein ring
    phi = np.linspace(
        0,
        2 * np.pi,
        500
    )

    axT.plot(
        np.cos(phi),
        np.sin(phi),
        "k--",
        lw=1.2,
        alpha=0.7,
        label=r"$\theta_E$"
    )

    # ---------------------------------------------------------
    # Sin paralaje
    # ---------------------------------------------------------

    axT.plot(
        x_no,
        y_no,
        lw=2,
        label="No parallax"
    )

    add_direction_arrows(
        axT,
        x_no,
        y_no,
        color="C0",
        n_arrows=4
    )

    # ---------------------------------------------------------
    # Con paralaje
    # ---------------------------------------------------------

    axT.plot(
        x_par,
        y_par,
        "--",
        lw=2,
        label="Parallax"
    )

    add_direction_arrows(
        axT,
        x_par,
        y_par,
        color="C1",
        n_arrows=4
    )

    # ---------------------------------------------------------
    # Lens
    # ---------------------------------------------------------

    axT.scatter(
        0,
        0,
        marker="+",
        s=100,
        color="k",
        linewidth=2,
        zorder=10,
        label="Lens"
    )

    # ========================================================
    # Mostrar tamaño de la fuente rho
    # ========================================================

    if show_source:

        # Punto de máximo acercamiento para cada trayectoria

        u_no = np.hypot(
            x_no,
            y_no
        )

        u_par = np.hypot(
            x_par,
            y_par
        )

        i_no = np.argmin(u_no)
        i_par = np.argmin(u_par)

        source_no = Circle(
            (
                x_no[i_no],
                y_no[i_no]
            ),
            radius=rho,
            fill=False,
            linewidth=1.5,
            alpha=0.8
        )

        source_par = Circle(
            (
                x_par[i_par],
                y_par[i_par]
            ),
            radius=rho,
            fill=False,
            linewidth=1.5,
            linestyle="--",
            alpha=0.8
        )

        axT.add_patch(
            source_no
        )

        axT.add_patch(
            source_par
        )

        axT.scatter(
            x_no[i_no],
            y_no[i_no],
            s=30,
            zorder=8
        )

        axT.scatter(
            x_par[i_par],
            y_par[i_par],
            s=30,
            zorder=8
        )

    # ---------------------------------------------------------
    # Punto correspondiente a t0
    # ---------------------------------------------------------

    i_t0 = np.argmin(
        np.abs(t - t0)
    )

    axT.scatter(
        x_no[i_t0],
        y_no[i_t0],
        marker="o",
        s=55,
        facecolors="none",
        edgecolors="C0",
        linewidths=1.5,
        zorder=10
    )

    axT.scatter(
        x_par[i_t0],
        y_par[i_t0],
        marker="s",
        s=55,
        facecolors="none",
        edgecolors="C1",
        linewidths=1.5,
        zorder=10
    )

    # ---------------------------------------------------------
    # Ejes
    # ---------------------------------------------------------

    axT.axhline(
        0,
        color="k",
        lw=0.6,
        alpha=0.25
    )

    axT.axvline(
        0,
        color="k",
        lw=0.6,
        alpha=0.25
    )

    axT.set_xlim(
        -traj_lim,
        traj_lim
    )

    axT.set_ylim(
        -traj_lim,
        traj_lim
    )

    axT.set_aspect(
        "equal",
        adjustable="box"
    )

    axT.set_xlabel(
        r"$u_x$"
    )

    axT.set_ylabel(
        r"$u_y$"
    )

    axT.set_title(
        "Source trajectory"
    )

    axT.legend(
        frameon=False,
        fontsize=9,
        loc="best"
    )

    axT.grid(
        alpha=0.15
    )

    # ---------------------------------------------------------
    # Información RA / Dec
    # ---------------------------------------------------------

    axT.text(
        0.03,
        0.03,

        (
            rf"$\mathrm{{RA}}={ra:.2f}^\circ$" "\n"
            rf"$\mathrm{{Dec}}={dec:.2f}^\circ$"
        ),

        transform=axT.transAxes,

        ha="left",
        va="bottom",

        fontsize=9,

        bbox=dict(
            boxstyle="round,pad=0.25",
            fc="white",
            ec="0.7",
            alpha=0.85
        )
    )

    plt.show()


# ============================================================
# SLIDERS
# ============================================================

style = {
    "description_width": "110px"
}

layout = widgets.Layout(
    width="370px"
)


sliders = dict(

    # ========================================================
    # FSPL
    # ========================================================

    t0=widgets.FloatSlider(
        value=2461000.0,
        min=2459000.0,
        max=2463000.0,
        step=1.0,
        description="t0 [JD]",
        style=style,
        layout=layout,
        continuous_update=False,
        readout_format=".1f"
    ),

    u0=widgets.FloatSlider(
        value=0.10,
        min=-1.0,
        max=1.0,
        step=0.005,
        description="u0",
        style=style,
        layout=layout,
        continuous_update=False,
        readout_format=".3f"
    ),

    tE=widgets.FloatLogSlider(
        value=100.0,
        base=10,
        min=0.0,       # 1 day
        max=3.0,       # 1000 days
        step=0.02,
        description="tE [days]",
        style=style,
        layout=layout,
        continuous_update=False,
        readout_format=".2f"
    ),

    rho=widgets.FloatLogSlider(
        value=0.01,
        base=10,
        min=-5.0,
        max=0.0,
        step=0.02,
        description="rho",
        style=style,
        layout=layout,
        continuous_update=False,
        readout_format=".2e"
    ),

    # ========================================================
    # PARALLAX
    # ========================================================

    piEN=widgets.FloatSlider(
        value=0.20,
        min=-2.0,
        max=2.0,
        step=0.01,
        description="piEN",
        style=style,
        layout=layout,
        continuous_update=False
    ),

    piEE=widgets.FloatSlider(
        value=0.10,
        min=-2.0,
        max=2.0,
        step=0.01,
        description="piEE",
        style=style,
        layout=layout,
        continuous_update=False
    ),

    # ========================================================
    # SKY POSITION
    # ========================================================

    ra=widgets.FloatSlider(
        value=170.0,
        min=0.0,
        max=360.0,
        step=1.0,
        description="RA [deg]",
        style=style,
        layout=layout,
        continuous_update=False
    ),

    dec=widgets.FloatSlider(
        value=-70.0,
        min=-90.0,
        max=90.0,
        step=1.0,
        description="Dec [deg]",
        style=style,
        layout=layout,
        continuous_update=False
    ),

    # ========================================================
    # PLOT
    # ========================================================

    window_k=widgets.FloatSlider(
        value=4.0,
        min=0.5,
        max=10.0,
        step=0.5,
        description="window [tE]",
        style=style,
        layout=layout,
        continuous_update=False
    ),

    n_points=widgets.IntSlider(
        value=4000,
        min=500,
        max=15000,
        step=500,
        description="N points",
        style=style,
        layout=layout,
        continuous_update=False
    ),

    traj_lim=widgets.FloatSlider(
        value=2.0,
        min=0.1,
        max=8.0,
        step=0.1,
        description="traj lim",
        style=style,
        layout=layout,
        continuous_update=False
    ),

    logA=widgets.Checkbox(
        value=False,
        description="log10(A)",
        indent=False,
        layout=widgets.Layout(
            width="180px"
        )
    ),

    show_source=widgets.Checkbox(
        value=True,
        description="show rho",
        indent=False,
        layout=widgets.Layout(
            width="180px"
        )
    )
)


# ============================================================
# ORGANIZACIÓN DE LA INTERFAZ
# ============================================================

ui_fspl = widgets.VBox([

    widgets.HTML(
        "<b>FSPL parameters</b>"
    ),

    sliders["t0"],
    sliders["u0"],
    sliders["tE"],
    sliders["rho"],
])


ui_parallax = widgets.VBox([

    widgets.HTML(
        "<b>Parallax</b>"
    ),

    sliders["piEN"],
    sliders["piEE"],

    widgets.HTML(
        "<br><b>Sky position</b>"
    ),

    sliders["ra"],
    sliders["dec"],
])


ui_plot = widgets.VBox([

    widgets.HTML(
        "<b>Plot options</b>"
    ),

    sliders["window_k"],
    sliders["n_points"],
    sliders["traj_lim"],
    sliders["logA"],
    sliders["show_source"],
])


ui = widgets.HBox([
    ui_fspl,
    ui_parallax,
    ui_plot
])


# ============================================================
# INTERACTIVE OUTPUT
# ============================================================

out_plot = widgets.interactive_output(
    plot_fspl_parallax_widget,
    sliders
)

display(
    ui,
    out_plot
)

Output()